<a href="https://colab.research.google.com/github/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/metabolomics/notebooks/01_metabolomics_preprocessing_metaboigniter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧫 Preprocessing Metabolomics with nf-core/metaboigniter
---

This session mirrors the proteomics one: raw mass-spectrometer output in, a table of
numbers out. The chemistry is different enough that the computational problem changes
shape completely, and understanding *why* is the point of the next 90 minutes.

### What you will be able to do afterwards

1. Explain the difference between **targeted** and **untargeted** metabolomics, and say
   which one your own project needs.
2. Open a raw file and see, with your own eyes, what a chromatogram and a peak area are.
3. Run [**nf-core/metaboigniter**](https://nf-co.re/metaboigniter/2.0.1/) end to end and
   read its outputs.
4. Judge the quality of a metabolomics matrix using **pooled QC samples**.

> 🧬 biology · ⚙️ technology · 💻 code — read whichever you need most.

## 🧬 Why measure metabolites at all?

The central dogma reads DNA → RNA → protein, and metabolites sit one step further still:
they are the **substrates and products** of what proteins do. That position has two
consequences.

- Metabolites are the layer **closest to the phenotype**. A change in the metabolome is
  often a change in physiology, not merely in potential.
- Metabolites are **fast**. Transcript levels shift over hours; metabolite pools shift
  over minutes. In an acute condition such as sepsis this is exactly the timescale of
  interest.

Sepsis is, among other things, a **metabolic** catastrophe. Tissue oxygen delivery fails,
cells switch to anaerobic glycolysis, lactate accumulates, amino-acid catabolism and
fatty-acid oxidation are reprogrammed to feed an activated immune system, and the liver
and kidney — the organs that clear metabolites — begin to fail. Measuring the serum
metabolome is a way of asking *how far along that cascade a patient is*.

In our cohort the authors found **128 metabolites** distinguishing CRKP from CSKP sepsis,
enriched in **thermogenesis** and **amino-acid / fatty-acid degradation** — and when
combined with the proteome, in **cysteine and methionine metabolism** and the
**folate-mediated one-carbon pool**. We will rediscover those results ourselves on Day 3.

> 🧬 A caution to carry through the whole course: metabolites are also exquisitely
> sensitive to things that have nothing to do with your hypothesis — when the patient last
> ate, which drugs they received, how long the tube sat on the bench before centrifugation.
> In a hospital cohort, some of what you measure is biology and some is logistics.

## ⚙️ Two philosophies: untargeted and targeted

|  | **Untargeted** (discovery) | **Targeted** (quantification) |
|---|---|---|
| Question | "What is different?" | "How much of *these* compounds?" |
| Instrument | High-resolution: Q-TOF, Orbitrap | Triple quadrupole (QqQ) |
| Acquisition | Full-scan MS1 (+ MS2 on selected ions) | **MRM**: monitor a fixed list of transitions |
| Coverage | Thousands of *features*, mostly unidentified | Hundreds to ~1 500 *known* compounds |
| Output unit | feature = (*m/z*, retention time) | named compound |
| Hard part | Feature detection, alignment, **annotation** | Building the transition library beforehand |
| Quantitative accuracy | Relative, semi-quantitative | Excellent, can be absolute with standards |

### Where our data sit

The course cohort was measured with **widely-targeted metabolomics**: an
**LC-ESI-MS/MS** system (SCIEX ExionLC AD + **QTRAP 6500**) running **MRM** against a
commercial library of pre-optimised transitions, in **both** positive and negative
ionisation mode, over a 10-minute gradient. The result is 1 073 **named** metabolites —
no annotation guesswork, and the compound identities come with KEGG, HMDB and PubChem
identifiers, which is a gift for the pathway analysis on Day 3.

### ⚠️ What that means for this notebook

**nf-core/metaboigniter is an untargeted pipeline.** Its whole job — detect peaks in
full-scan data, group them into features, align retention times across samples — assumes
full-scan spectra. Our MRM files contain **no spectra at all**, only pre-selected
chromatograms, so metaboigniter cannot process them. That is not a limitation of the
pipeline or of the data; the two simply answer different questions.

So this notebook does both halves honestly:

1. **First** we open the cohort's real MRM files and watch a peak area being measured —
   that is where the numbers in our matrix come from.
2. **Then** we run metaboigniter on a small untargeted dataset, because untargeted
   LC-MS is what most of you will meet in your own projects, and because the pipeline
   concepts transfer directly.

The dataset used in part 2 is a **placeholder**, marked clearly below, and is designed to
be swapped for another untargeted study by editing a single configuration cell.

## 💻 Part 1 — inside a real raw file

We will read one deposited file from [MTBLS14016](https://www.ebi.ac.uk/metabolights/MTBLS14016)
using nothing but the Python standard library. An **mzML** file is XML with the numeric
arrays base64-encoded and usually zlib-compressed; opening it by hand once demystifies the
format for good.

In [ ]:
COURSE_REPO = "Multiomics-Analytics-Group/course_multi-omics_analysis"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/{BRANCH}"

# Raw files are ~4 MB each and are not kept in the course repository — we fetch them
# straight from MetaboLights. The path comes from the 'Derived Spectral Data File' column
# of the study's own assay table, so it is the study's declaration of where its files live.
MTBLS_STUDY = "MTBLS14016"
MTBLS_FILES = (
    f"https://ftp.ebi.ac.uk/pub/databases/metabolights/studies/public/{MTBLS_STUDY}"
    "/FILES/DERIVED_FILES"
)

RAW_FILE = "Con10_P.mzML"  # sample Con10, positive ionisation mode
print("Reading:", RAW_FILE, "\nfrom:", f"{MTBLS_FILES}/{RAW_FILE}")

In [ ]:
!mkdir -p metabolomics/data/raw
!wget -q -nc {MTBLS_FILES}/{RAW_FILE} -O metabolomics/data/raw/{RAW_FILE}
!ls -lh metabolomics/data/raw/

In [ ]:
import base64
import struct
import zlib
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd

NS = {"mz": "http://psi.hupo.org/ms/mzml"}


def _decode(binary_array: ET.Element) -> np.ndarray:
    """Decode one <binaryDataArray>: base64 -> (optional zlib) -> float array."""
    params = {p.get("name") for p in binary_array.findall("mz:cvParam", NS)}
    raw = base64.b64decode(binary_array.findtext("mz:binary", default="", namespaces=NS))
    if "zlib compression" in params:
        raw = zlib.decompress(raw)
    fmt = "d" if "64-bit float" in params else "f"
    values = struct.unpack(f"<{len(raw) // struct.calcsize(fmt)}{fmt}", raw)
    return np.asarray(values)


def read_chromatograms(path: str) -> list[dict]:
    """Return every chromatogram in an mzML file as {id, q1, time, intensity}."""
    out = []
    for _, chrom in ET.iterparse(path, events=("end",)):
        if not chrom.tag.endswith("}chromatogram"):
            continue
        arrays = {}
        for arr in chrom.findall(".//mz:binaryDataArray", NS):
            names = {p.get("name") for p in arr.findall("mz:cvParam", NS)}
            # Files may carry extra vendor arrays; keep only the two standard ones.
            if "time array" in names:
                arrays["time"] = _decode(arr)
            elif "intensity array" in names:
                arrays["intensity"] = _decode(arr)
        target = chrom.find(".//mz:precursor//mz:cvParam[@name='isolation window target m/z']", NS)
        out.append(
            {
                "id": chrom.get("id"),
                "q1": float(target.get("value")) if target is not None else np.nan,
                "time": arrays.get("time"),
                "intensity": arrays.get("intensity"),
            }
        )
        chrom.clear()
    return out


chromatograms = read_chromatograms(f"metabolomics/data/raw/{RAW_FILE}")
print(f"{len(chromatograms)} chromatograms in the file")
print("first three:", [c["id"][:60] for c in chromatograms[:3]])

## How MRM Works

**First Stage (Q1)**: The mass spectrometer isolates a specific precursor ion of the target metabolite.

**Fragmentation (Q2/Collision Cell)**: The selected ion is broken down into characteristic product ions.

**Second Stage (Q3)**: A specific product ion resulting from the breakdown is filtered and detected.

**Ion Transition**: The specific pair of precursor and product ions forms a unique signature (transition) monitored on a millisecond timescale.

Three things are worth noticing.

**There are no spectra.** A triple quadrupole is not a scanning instrument here: the first
quadrupole is parked on one precursor mass (**Q1**), the second fragments it, the third is
parked on one fragment mass (**Q3**). That pair is a **transition**, and what you record
over time is a single-ion chromatogram. About 1 500 of them, cycled through as the
gradient runs.

**The first two are TIC and BPC** — total and base-peak ion current, useful summaries of
whether the run behaved.

**Only Q1 survives conversion.** The vendor-to-mzML conversion here did not carry the Q3
masses across, so transitions sharing a precursor are distinguished only by an index. This
is the kind of small metadata loss that makes reprocessing public data harder than it
should be — worth remembering when you deposit your own.

In [ ]:
import matplotlib.pyplot as plt

tic = next(c for c in chromatograms if c["id"] == "TIC")
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(tic["time"], tic["intensity"], lw=0.8, color="steelblue")
ax.set(
    xlabel="Retention time (min)",
    ylabel="Total ion current",
    title=f"TIC — {RAW_FILE}: the whole 10-minute gradient in one line",
)
fig.tight_layout()

### A peak, and its area

Now the single most important picture in quantitative metabolomics. Below are a few SRM
traces. Each bump is a compound eluting; the **area under the bump** is the number that
ends up in our data matrix.

In [ ]:
srm = [c for c in chromatograms if c["id"].startswith(("SRM", "- SRM")) and c["intensity"] is not None]
srm_sorted = sorted(srm, key=lambda c: c["intensity"].max(), reverse=True)

fig, axes = plt.subplots(2, 2, figsize=(11, 5.5), sharex=True)
for ax, chrom in zip(axes.ravel(), srm_sorted[:4]):
    time, signal = chrom["time"], chrom["intensity"]
    area = np.trapezoid(signal, time) if hasattr(np, "trapezoid") else np.trapz(signal, time)
    ax.fill_between(time, signal, color="steelblue", alpha=0.35)
    ax.plot(time, signal, color="steelblue", lw=0.9)
    ax.set_title(f"Q1 = {chrom['q1']:.2f}   |   area = {area:.3g}", fontsize=9)
    ax.set_xlabel("RT (min)")
    ax.set_ylabel("Intensity")
fig.suptitle("Four of the most intense MRM transitions, with integrated areas", y=1.02)
fig.tight_layout()

That is the entire quantification step: **integrate the peak, record the area, repeat for
every transition in every sample**. Vendor software (here SCIEX Analyst / MultiQuant)
does it with peak-shape models and manual review; the arithmetic is no more mysterious
than the `np.trapezoid` call above.

The judgement calls are all in the details: where does the peak start and stop, is that
shoulder a co-eluting isomer, is the signal above the noise at all. Those decisions are
why a targeted metabolomics core facility employs people rather than only computers.

> 💻 In production, use [pyOpenMS](https://pyopenms.readthedocs.io/) rather than the
> hand-rolled parser above:
> `MzMLFile().load(path, exp)` then `exp.getChromatograms()`. Our version exists to show
> you that there is no magic inside the file.

## ⚙️ Part 2 — the untargeted workflow

When you *do* have full-scan data, the raw file is a three-dimensional cloud: intensity
over (*m/z*, retention time). Turning that cloud into a table is a longer road than
integrating a known peak, and every step has failure modes:

| Step | What it does | What goes wrong |
|---|---|---|
| **Centroiding / peak picking** | reduce each raw profile peak to one *m/z* + intensity | over-smoothing loses low-abundance ions |
| **Mass-trace detection** | follow one *m/z* across consecutive scans | traces split, or merge with neighbours |
| **Feature detection** (`FeatureFinderMetabo`) | assemble traces + isotope patterns into features | isotopes of one compound counted as separate features |
| **Adduct deconvolution** (`MetaboliteAdductDecharger`) | recognise [M+H]⁺, [M+Na]⁺, [M+NH₄]⁺ … as one molecule | one metabolite inflated into five features |
| **Retention-time alignment** | correct drift between injections | over-warping invents alignment that is not there |
| **Linking** | match features across samples into one consensus row | mismatches create fake missing values |
| **Requantification** | go back to the raw data for features missing in some samples | recovers real zeros *and* real noise |
| **Identification** (SIRIUS, MS2Query, GNPS) | put a name on a feature | the honest answer is often "unknown" |

The last row deserves emphasis. In a typical untargeted study **fewer than 20 %** of
features are confidently identified. A perfectly real, highly significant feature can stay
a nameless (*m/z*, RT) pair — which is precisely why our targeted cohort, with 1 073 named
compounds, is such a comfortable dataset to learn integration on.

`nf-core/metaboigniter` v2.0.1 wires those steps together on top of
[OpenMS](https://www.openms.de/), with optional SIRIUS/CSI:FingerID and MS2Query
identification.

## 💻 Setting up Nextflow

Java and Nextflow install exactly as in the proteomics session. Two things differ here,
and both are worth a moment.

**The execution profile.** `quantmsdiann`/DIA-NN forced us into a container (Docker or
Apptainer), but `metaboigniter` supports **Conda**, so here we skip container engines
altogether and let Nextflow manage a Conda environment per process instead.

**The config parser.** Nextflow 26.04 made its new *strict syntax* parser the default.
`metaboigniter` 2.0.1 was released over two years ago, before that change, and its
`nextflow.config` defines a Groovy helper function (`check_max`) that the strict parser
refuses to read:

```
Config parsing failed
Error nextflow.config:475:14: Unexpected input: '('
```

Since 2.0.1 is the latest release, that will not be fixed upstream, so the setup cell sets
`NXF_SYNTAX_PARSER=v1` to request the legacy parser. Note that this is a transitional
escape hatch — Nextflow intends to remove the legacy parser eventually
([migration guide](https://nf-co.re/docs/developing/migration-guides/strict-syntax)).

> ⚠️ Do **not** solve this by pinning an older Nextflow: `quantmsdiann` declares
> `nextflowVersion = '!>=25.10.4'` and enforces it, so the proteomics session needs a
> recent Nextflow. One current Nextflow plus the legacy-parser flag here is the only
> combination that satisfies both pipelines.


In [ ]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
!wget -qO- https://get.nextflow.io | bash
!mv -f nextflow /usr/local/bin/nextflow && chmod +x /usr/local/bin/nextflow
!nextflow -v

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

MINIFORGE_PREFIX = Path("/usr/local/miniforge3")
MINIFORGE_BIN = MINIFORGE_PREFIX / "bin"

if not shutil.which("conda") and not (MINIFORGE_BIN / "conda").exists():
    print("Installing Miniforge (conda) ...")
    subprocess.run(
        "wget -q https://github.com/conda-forge/miniforge/releases/latest/download/"
        "Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh",
        shell=True,
    )
    subprocess.run(f"bash /tmp/miniforge.sh -b -p {MINIFORGE_PREFIX}", shell=True)

if str(MINIFORGE_BIN) not in os.environ["PATH"]:
    os.environ["PATH"] = f"{MINIFORGE_BIN}:{os.environ['PATH']}"

# nf-core pipelines pull most of their tools from bioconda, but Miniforge ships with
# conda-forge only. Same channel setup as the official nf-core Colab guide:
# https://nf-co.re/docs/tutorials/google_colab/nf-core_colab_guide
for channel in ("bioconda", "conda-forge"):
    subprocess.run(f"conda config --add channels {channel}", shell=True, capture_output=True)
subprocess.run("conda config --set channel_priority strict", shell=True, capture_output=True)

# metaboigniter 2.0.1 predates Nextflow's "strict syntax" config parser, which became the
# default in Nextflow 26.04. Its nextflow.config defines a Groovy helper (`check_max`) that
# the strict parser rejects with "Config parsing failed ... Unexpected input: '('".
# The pipeline is unmaintained (2.0.1 is the latest release), so we ask Nextflow for the
# legacy parser instead of waiting for a fix.
os.environ["NXF_SYNTAX_PARSER"] = "v1"

EXECUTION_PROFILE = "conda" if shutil.which("conda") else None
print("Execution profile:", EXECUTION_PROFILE, "—", shutil.which("conda"))


> 💡 **Why Conda here, and not a container.** `quantmsdiann` needed Docker/Apptainer
> because DIA-NN has no Conda package — and Apptainer's own overlay mount can itself fail
> on Colab (see the `CAP_DAC_READ_SEARCH` workaround in the proteomics notebook).
> `metaboigniter` has no such restriction, so the cell above installs a small
> [Miniforge](https://github.com/conda-forge/miniforge) distribution instead — no
> container engine, no extra apt repositories, no kernel restart. Nextflow then builds a
> Conda environment per process the first time it needs one.
>
> The tradeoff: Conda environment resolution is slower than pulling a pre-built container
> image, and marginally less reproducible (a fresh solve can, in principle, pick different
> transitive dependency versions over time). Fine for a one-off teaching run; for a
> production pipeline, nf-core still recommends containers where they're available.


## 📋 The samplesheet

metaboigniter's metadata is much simpler than SDRF — a four-column CSV:

| Column | Meaning |
|---|---|
| `sample` | unique name for the injection |
| `level` | `MS1`, `MS2` or `MS12` — which acquisition level this file contains |
| `type` | the group label: your conditions, plus `QC_POOL` for pooled quality controls |
| `msfile` | path to an **indexed** `.mzML` |

Two traps worth knowing before you lose an afternoon to them:

- **`.mzML` must be indexed.** metaboigniter relies on the index rather than rebuilding
  it; unindexed files fail with an opaque error. Fix with
  [`bin/reindex_mzml.py`](https://github.com/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/bin/reindex_mzml.py).
- **MS2 files are declared separately** and linked to their MS1 run through the `type` and
  `--ms2_collection_model` settings (`paired` if MS2 was collected on the same injections,
  `separate` if in dedicated runs).

## 🚀 A first run: the pipeline's own test data

The fastest way to know your environment works is the built-in `test` profile, which
points at a miniature dataset hosted by nf-core.

```bash
nextflow run nf-core/metaboigniter -r 2.0.1 \
    -profile test,conda \
    --outdir metabolomics/results/test_run
```


In [ ]:
if EXECUTION_PROFILE:
    command = (
        "nextflow run nf-core/metaboigniter -r 2.0.1 "
        f"-profile test,{EXECUTION_PROFILE} "
        "--outdir metabolomics/results/test_run -resume"
    )
    print(command, "\n")
    !{command}
else:
    print("Skipped: no execution profile available in this runtime.")


In [ ]:
!find metabolomics/results/test_run -maxdepth 2 2>/dev/null | sort | head -40

The file to look for is the **quantification table** — features in rows, samples in
columns, plus *m/z*, retention time, charge and adduct annotation. Under
`.../linking/` you will find `*Linked_data.tsv`; that is the metabolomics equivalent of
`report.pg_matrix.tsv`, and the input to the analysis notebook.

In [ ]:
import glob

candidates = sorted(glob.glob("metabolomics/results/test_run/**/*Linked_data.tsv", recursive=True))
print("quantification tables found:", candidates)
if candidates:
    table = pd.read_csv(candidates[0], sep="\t", index_col=0)
    print(f"{table.shape[0]} features x {table.shape[1]} columns")
    display(table.head())

## 🔄 Running it on a real study — the swappable part

### ⚠️ Placeholder dataset

The block below runs metaboigniter on
[**MTBLS8735**](https://www.ebi.ac.uk/metabolights/MTBLS8735), the *Metabonaut* example
study: three cardiovascular-disease patients, three healthy controls and four pooled QC
injections, acquired in positive mode with separate MS2 runs. It is small, public,
genuinely untargeted, and has the same two-groups-plus-QC shape as our course cohort — so
every analytical idea transfers.

**It is a stand-in.** To point this notebook at a different untargeted study, change only
the configuration cell below: the file list, the group labels, and the polarity. Nothing
further down depends on which study it is.

In [ ]:
# ---------------------------------------------------------------------------------------
# DATASET CONFIGURATION — edit this cell to use a different untargeted study
# ---------------------------------------------------------------------------------------
DATASET = "MTBLS8735"
FTP_BASE = f"https://ftp.ebi.ac.uk/pub/databases/metabolights/studies/public/{DATASET}/FILES"
POLARITY = "positive"

# sample name -> (acquisition level, group label, remote file name)
FILES = {
    "MS_A_POS": ("MS1", "CVD", "MS_A_POS.mzML"),
    "MS_B_POS": ("MS1", "CTR", "MS_B_POS.mzML"),
    "MS_D_POS": ("MS1", "CVD", "MS_D_POS.mzML"),
    "MS_E_POS": ("MS1", "CTR", "MS_E_POS.mzML"),
    "MS_QC_POOL_1_POS": ("MS1", "QC_POOL", "MS_QC_POOL_1_POS.mzML"),
    "MS_QC_POOL_2_POS": ("MS1", "QC_POOL", "MS_QC_POOL_2_POS.mzML"),
}
# ---------------------------------------------------------------------------------------

data_dir = f"metabolomics/data/{DATASET}"
samplesheet = pd.DataFrame(
    [
        {"sample": name, "level": level, "type": group, "msfile": f"{data_dir}/{fname}"}
        for name, (level, group, fname) in FILES.items()
    ]
)
samplesheet

We deliberately use a **subset** — four biological samples and two QC injections instead
of the full study. Feature detection and alignment scale with the number of files, and a
subset keeps the run inside a coffee break. For a real analysis you would of course use
every sample: alignment and linking get *better* with more runs, not worse.

In [ ]:
import os

os.makedirs(data_dir, exist_ok=True)
for _, group, fname in FILES.values():
    if not os.path.exists(f"{data_dir}/{fname}"):
        !wget -q -nc {FTP_BASE}/{fname} -O {data_dir}/{fname}
samplesheet.to_csv(f"{data_dir}/samplesheet.csv", index=False)
!ls -lh {data_dir}

### Make sure the mzML files are indexed

In [ ]:
!wget -q -nc {BASE_URL}/bin/reindex_mzml.py -O reindex_mzml.py
!pip install -q pyopenms 2>/dev/null | tail -1
!python reindex_mzml.py --input_dir {data_dir} || echo "Re-indexing skipped (pyopenms unavailable)"

### Pipeline parameters

Defaults tuned for a Q-TOF are rarely right for your instrument. The config below comes
from the BRIGHT metabolomics course and is a sane starting point for high-resolution
Q-TOF data; the comments say what each knob controls. **Do not copy parameters between
instruments without checking them.**

In [ ]:
config = """
params {
    // --- peak picking -----------------------------------------------------------------
    algorithm_signal_to_noise_peakpickerhires_openms                      = 0.1
    algorithm_spacing_difference_gap_peakpickerhires_openms               = 4.0
    algorithm_signaltonoise_auto_max_stdev_factor_peakpickerhires_openms  = 3.0

    // --- feature detection: what counts as a real mass trace --------------------------
    algorithm_common_noise_threshold_int_featurefindermetabo_openms       = 60.0
    algorithm_common_chrom_peak_snr_featurefindermetabo_openms            = 3.0
    algorithm_epd_masstrace_snr_filtering_featurefindermetabo_openms      = true
    algorithm_ffm_charge_upper_bound_featurefindermetabo_openms           = 1

    // --- retention-time alignment tolerances (seconds / ppm) --------------------------
    algorithm_max_num_peaks_considered_mapalignerposeclustering_openms    = 2000
    algorithm_pairfinder_distance_rt_max_difference_mapalignerposeclustering_openms = 45

    // --- linking features across samples ---------------------------------------------
    algorithm_warp_rt_tol_featurelinkerunlabeledkd_openms                 = 45.0
    algorithm_warp_mz_tol_featurelinkerunlabeledkd_openms                 = 3.0
    algorithm_link_rt_tol_featurelinkerunlabeledkd_openms                 = 12.0
    algorithm_link_mz_tol_featurelinkerunlabeledkd_openms                 = 6.0
}

process {
    resourceLimits = [ cpus: 2, memory: '12.GB', time: '6.h' ]
}
"""
with open("metabolomics/metaboigniter.config", "w") as handle:
    handle.write(config)
print(config)

In [ ]:
if EXECUTION_PROFILE:
    command = (
        "nextflow run nf-core/metaboigniter -r 2.0.1 "
        f"-profile {EXECUTION_PROFILE} "
        f"-c metabolomics/metaboigniter.config "
        f"--input {data_dir}/samplesheet.csv "
        f"--polarity {POLARITY} "
        "--requantification "
        f"--outdir metabolomics/results/{DATASET} -resume"
    )
    print(command, "\n")
    !{command}
else:
    print("Skipped: no execution profile available in this runtime.")


> ⏱️ Adding `--identification --run_ms2query` turns on annotation, which downloads about
> **2 GB** of GNPS spectral-library models on first use. Worth doing on a real project.

In [ ]:
candidates = sorted(glob.glob(f"metabolomics/results/{DATASET}/**/*Linked_data.tsv", recursive=True))
print("quantification tables:", candidates)
if candidates:
    features = pd.read_csv(candidates[0], sep="\t", index_col=0)
    print(f"{features.shape[0]} features across {features.shape[1]} columns")
    display(features.head())

## 📈 The course cohort's metabolite matrix

Back to our patients. This is the table produced from the MRM peak areas — one row per
named metabolite, one column per injection — and the object of Day 3's first session.

In [ ]:
metabolites = pd.read_csv(f"{BASE_URL}/metabolomics/data/metabolite_matrix.tsv", sep="\t")
annotation = pd.read_csv(f"{BASE_URL}/metabolomics/data/metabolite_annotation.tsv", sep="\t")
metadata = pd.read_csv(f"{BASE_URL}/metadata/sample_metadata.tsv", sep="\t")

info_cols = ["metabolite", "formula", "q1_mz", "adduct", "ion_mode"]
qc_cols = [c for c in metabolites.columns if c.startswith("QC")]
patient_cols = [c for c in metabolites.columns if c not in info_cols + qc_cols]

print(f"{metabolites.shape[0]} metabolites x {len(patient_cols)} patients + {len(qc_cols)} pooled QC")
metabolites[info_cols + patient_cols[:4]].head()

### What kind of molecules are these?

In [ ]:
counts = annotation["class_i"].value_counts()
ax = counts.sort_values().plot(kind="barh", figsize=(8, 4.5), color="steelblue")
ax.set(
    xlabel="Number of annotated metabolites",
    title="Compound classes among the differential metabolites",
)
ax.figure.tight_layout()
print(f"{annotation['kegg_compound'].notna().sum()} of {len(annotation)} carry a KEGG compound ID")

Amino acids and their metabolites, fatty acids (FA), organic acids and
glycerophospholipids (GP) dominate — the classes you would expect a library built for
human plasma to cover, and the classes most relevant to sepsis metabolism.

The KEGG identifiers are what make Day 3 possible: they let us map metabolites and
proteins onto the *same* pathways.

### Ionisation mode, and why it matters

A molecule is only seen if it ionises. Positive mode favours basic groups (amines,
amino acids); negative mode favours acidic ones (carboxylic acids, phosphates). Running
both is not redundancy — it is coverage.

In [ ]:
metabolites["ion_mode"].value_counts().rename("metabolites").to_frame()

### Quality control with pooled samples

Six aliquots of the *same* pooled serum were injected through the batch. For each
metabolite we can therefore separate **technical** from **biological** variability:

$$\text{CV} = \frac{\sigma}{\mu} \qquad \text{computed across QC injections}$$

The paper kept metabolites with **CV < 0.3** in the QCs. Let us check that ourselves —
reproducing a published filtering step is one of the most useful things you can do with
somebody else's data.

In [ ]:
qc = metabolites[qc_cols]
cv_qc = qc.std(axis=1) / qc.mean(axis=1)
bio = metabolites[patient_cols]
cv_bio = bio.std(axis=1) / bio.mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(cv_qc.dropna(), bins=60, color="steelblue")
axes[0].axvline(0.3, color="firebrick", ls="--", label="CV = 0.3 (paper's cut-off)")
axes[0].set(xlabel="CV across pooled QC injections", ylabel="Metabolites",
            title="Technical variability")
axes[0].legend()
axes[1].scatter(cv_qc, cv_bio, s=6, alpha=0.35, color="steelblue")
lim = [0, max(cv_qc.max(), cv_bio.max())]
axes[1].plot(lim, lim, color="grey", ls=":")
axes[1].set(xlabel="CV in QC (technical)", ylabel="CV in patients (biological + technical)",
            title="Signal vs noise, per metabolite")
fig.tight_layout()

print(f"metabolites with QC CV < 0.30 : {(cv_qc < 0.30).sum()} of {cv_qc.notna().sum()}")
print(f"median QC CV                  : {cv_qc.median():.3f}")
print(f"median patient CV             : {cv_bio.median():.3f}")

Read the right-hand panel carefully, because it is the single most informative plot in
metabolomics QC. Points **above the diagonal** vary more between patients than between
repeated injections of the same pool — those carry biological signal. Points **on or below**
the diagonal are dominated by measurement noise and should be filtered out before testing.

### Missingness

In [ ]:
print(f"missing values in the patient block: {bio.isna().to_numpy().mean():.2%}")
observed = bio.notna().sum(axis=1)
ax = observed.value_counts().sort_index().plot(
    kind="bar", figsize=(9, 3.2), color="steelblue"
)
ax.set(xlabel="Number of patients the metabolite was measured in", ylabel="Metabolites",
       title="Data completeness")
ax.figure.tight_layout()

Far more complete than the proteomics matrix (~5 % vs ~27 % missing) — the direct benefit
of a targeted method: you look for the same 1 073 compounds in every sample, so a missing
value means "below the limit of quantification" rather than "not selected for
fragmentation this time". That distinction changes how you should impute, and we will come
back to it on Day 3.

## 📚 Further reading

- nf-core/metaboigniter — <https://nf-co.re/metaboigniter/2.0.1/>
- The Metabonaut tutorial (untargeted LC-MS in R) — <https://rformassspectrometry.github.io/Metabonaut/>
- Broeckling *et al.* (2023) *Current and future perspectives on the structured
  annotation of untargeted metabolomics data.* Metabolites 13:1043.
- Dunn *et al.* (2011) *Procedures for large-scale metabolic profiling of serum and plasma.*
  Nat Protoc 6:1060–1083. — the origin of much of the QC practice above.